# <b>1. Data Splits.</b>

This notebook prepares the train/test splits used throughout the project.

The notebook performs the following steps:
- loads the source dataset;
- constructs rolling time windows (`W1`, `W2`, ...);
- splits each window into **train_model**, **calibration**, and **test** sets without training any models;
- saves the resulting datasets in **Parquet** format;
- verifies the correctness of the generated splits (time ordering, absence of overlap, dataset sizes, and saved files).

The output of this notebook is a collection of ready-to-use train/calibration/test splits that serve as the input for all subsequent modeling and evaluation notebooks.

In [ ]:
import sys
import gc
from pathlib import Path

import polars as pl
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd() / "src"))

from meta import load_split_dataset
from splits import get_window_ids, iter_window_splits

In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"X does not have valid feature names, but LGBMClassifier was fitted with feature names",
    category=UserWarning,
    module=r"sklearn\.utils\.validation"
)

## Config

In [3]:
INSTRUMENT = "btc"                 # "btc" or "eth"

WINDOWS_TO_RUN = get_window_ids()  # ["W1", "W2", ...]
OVERWRITE = True

DATA_PATH = Path(f"data/enriched/{INSTRUMENT}_sub40.parquet")

# Новая папка без meta-фичей.
SPLIT_DIR = Path(f"data/meta/{INSTRUMENT}_splits_only")

CALIB_FRAC = 0.20

print(f"Instrument : {INSTRUMENT}")
print(f"Windows    : {WINDOWS_TO_RUN}")
print(f"Data       : {DATA_PATH}")
print(f"Split dir  : {SPLIT_DIR}")
print(f"Calib frac : {CALIB_FRAC}")

Instrument : btc
Windows    : ['W1', 'W2', 'W3', 'W4', 'W5']
Data       : data\enriched\btc_sub40.parquet
Split dir  : data\meta\btc_splits_only
Calib frac : 0.2


## Run

In [4]:
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.height:,} rows, {len(df.columns)} cols")

Loaded 20,100,882 rows, 25 cols


In [ ]:
def save_plain_split_dataset(
    df: pl.DataFrame,
    output_dir: Path,
    instrument: str,
    window_ids: list[str],
    calib_frac: float = 0.20,
    overwrite: bool = False
):
    """
    Create and save train_model / calib / test parquet files
    without training any meta-models.

    Output files:
        {instrument}_{window_id}_train_model.parquet
        {instrument}_{window_id}_calib.parquet
        {instrument}_{window_id}_test.parquet
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    saved_rows = []

    split_iter = iter_window_splits(
        df=df,
        window_ids=window_ids,
        calib_frac=calib_frac,
    )

    for split in tqdm(list(split_iter), desc=f"plain splits {instrument}"):
        wid = split.window_id

        files = {
            "train_model": output_dir / f"{instrument}_{wid}_train_model.parquet",
            "calib":       output_dir / f"{instrument}_{wid}_calib.parquet",
            "test":        output_dir / f"{instrument}_{wid}_test.parquet",
        }

        if (not overwrite) and all(p.exists() for p in files.values()):
            print(f"{wid}: skip, files already exist")
            continue

        split.train_model.write_parquet(
            files["train_model"],
            compression="zstd",
            statistics=True,
        )

        split.calib.write_parquet(
            files["calib"],
            compression="zstd",
            statistics=True,
        )

        split.test.write_parquet(
            files["test"],
            compression="zstd",
            statistics=True,
        )

        print(f"\n{wid}")
        print(f"  saved {files['train_model'].name:30s} rows={split.train_model.height:,}")
        print(f"  saved {files['calib'].name:30s} rows={split.calib.height:,}")
        print(f"  saved {files['test'].name:30s} rows={split.test.height:,}")

        saved_rows.append({
            "window": wid,
            "train_model_rows": split.train_model.height,
            "calib_rows": split.calib.height,
            "test_rows": split.test.height,
            "train_model_file": str(files["train_model"]),
            "calib_file": str(files["calib"]),
            "test_file": str(files["test"]),
        })

        del split
        gc.collect()

    return pl.DataFrame(saved_rows)


df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.height:,} rows, {len(df.columns)} cols")

split_summary = save_plain_split_dataset(
    df=df,
    output_dir=SPLIT_DIR,
    instrument=INSTRUMENT,
    window_ids=WINDOWS_TO_RUN,
    calib_frac=CALIB_FRAC,
    overwrite=OVERWRITE,
)

display(split_summary)

del df
gc.collect()

Loaded 20,100,882 rows, 25 cols


plain splits btc:   0%|          | 0/5 [00:00<?, ?it/s]


W1
  saved btc_W1_train_model.parquet     rows=3,285,262
  saved btc_W1_calib.parquet           rows=424,282
  saved btc_W1_test.parquet            rows=1,971,880

W2
  saved btc_W2_train_model.parquet     rows=5,739,389
  saved btc_W2_calib.parquet           rows=720,356
  saved btc_W2_test.parquet            rows=2,068,316

W3
  saved btc_W3_train_model.parquet     rows=7,275,187
  saved btc_W3_calib.parquet           rows=1,719,093
  saved btc_W3_test.parquet            rows=2,766,251

W4
  saved btc_W4_train_model.parquet     rows=9,873,402
  saved btc_W4_calib.parquet           rows=3,883,705
  saved btc_W4_test.parquet            rows=2,804,940

W5
  saved btc_W5_train_model.parquet     rows=13,874,932
  saved btc_W5_calib.parquet           rows=3,786,672
  saved btc_W5_test.parquet            rows=1,858,399


window,train_model_rows,calib_rows,test_rows,train_model_file,calib_file,test_file
str,i64,i64,i64,str,str,str
"""W1""",3285262,424282,1971880,"""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…"
"""W2""",5739389,720356,2068316,"""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…"
"""W3""",7275187,1719093,2766251,"""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…"
"""W4""",9873402,3883705,2804940,"""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…"
"""W5""",13874932,3786672,1858399,"""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…","""data\meta\btc_splits_only\btc_…"


51

## Validation

In [6]:
for wid in WINDOWS_TO_RUN:
    split = load_split_dataset(SPLIT_DIR, INSTRUMENT, wid)

    tm_max  = split.train_model["timestamp"].max()
    cal_min = split.calib["timestamp"].min()
    cal_max = split.calib["timestamp"].max()
    tst_min = split.test["timestamp"].min()

    assert tm_max < cal_min, f"{wid}: train_model overlaps calib"
    assert cal_max < tst_min, f"{wid}: calib overlaps test"

    print(f"\n{wid}")
    print(
        f"  rows: "
        f"train_model={split.train_model.height:,} "
        f"calib={split.calib.height:,} "
        f"test={split.test.height:,}"
    )

    meta_cols = [c for c in split.train_model.columns if c.startswith("meta_")]
    print(f"  meta columns: {meta_cols}")

    del split
    gc.collect()


W1
  rows: train_model=3,285,262 calib=424,282 test=1,971,880
  meta columns: []

W2
  rows: train_model=5,739,389 calib=720,356 test=2,068,316
  meta columns: []

W3
  rows: train_model=7,275,187 calib=1,719,093 test=2,766,251
  meta columns: []

W4
  rows: train_model=9,873,402 calib=3,883,705 test=2,804,940
  meta columns: []

W5
  rows: train_model=13,874,932 calib=3,786,672 test=1,858,399
  meta columns: []


In [7]:
for p in sorted(SPLIT_DIR.glob(f"{INSTRUMENT}_*.parquet")):
    print(f"  {p.name:40s}  {p.stat().st_size / 1024**2:8.1f} MB")

  btc_W1_calib.parquet                          19.8 MB
  btc_W1_test.parquet                           92.1 MB
  btc_W1_train_model.parquet                   155.3 MB
  btc_W2_calib.parquet                          33.1 MB
  btc_W2_test.parquet                           94.7 MB
  btc_W2_train_model.parquet                   270.7 MB
  btc_W3_calib.parquet                          78.8 MB
  btc_W3_test.parquet                          127.8 MB
  btc_W3_train_model.parquet                   341.1 MB
  btc_W4_calib.parquet                         180.6 MB
  btc_W4_test.parquet                          128.1 MB
  btc_W4_train_model.parquet                   461.4 MB
  btc_W5_calib.parquet                         173.4 MB
  btc_W5_test.parquet                           83.5 MB
  btc_W5_train_model.parquet                   647.4 MB
